# GeoTIFF pseudo-site sampling demo

This notebook demonstrates how to:
- Build pseudo-sites from sample coordinates
- Sample environmental covariates from a GeoTIFF
- Train the multi-species resistance model
- Visualize learned resistances


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import tempfile

import numpy as np
import torch
import rasterio
from rasterio.transform import from_origin

ROOT = os.path.dirname(os.path.dirname(os.getcwd())) if os.getcwd().endswith('notebooks') else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, 'src'))

from multispecies_resistance.data import SpeciesData
from multispecies_resistance.train import build_species_graphs, train_model

torch.set_default_dtype(torch.float64)


In [ ]:
# Simulate samples and genotypes
rng = np.random.default_rng(7)
num_sites = 25
num_snps = 1500
num_env = 1

coords = np.column_stack([
    rng.uniform(34.0, 48.0, size=num_sites),
    rng.uniform(-120.0, -100.0, size=num_sites),
])

env = rng.normal(0.0, 1.0, size=(num_sites, num_env))
feats = np.concatenate([coords, env], axis=1)
weights = rng.normal(0.0, 0.5, size=(feats.shape[1], num_snps))
logits = feats @ weights + rng.normal(0.0, 0.5, size=(num_sites, num_snps))
freq = 1.0 / (1.0 + np.exp(-logits))
freq = np.clip(freq, 0.01, 0.99)

genotypes = []
sample_coords = []
for s in range(num_sites):
    n = int(rng.integers(5, 11))
    g = rng.binomial(2, freq[s], size=(n, num_snps))
    genotypes.append(g)
    sample_coords.append(np.repeat(coords[s][None, :], n, axis=0))

genotypes = np.vstack(genotypes)
sample_coords = np.vstack(sample_coords)
genotypes.shape, sample_coords.shape


In [ ]:
# Create a synthetic GeoTIFF env layer
tmp_dir = tempfile.mkdtemp()
raster_path = os.path.join(tmp_dir, 'env.tif')

height, width = 200, 200
transform = from_origin(-125.0, 50.0, 0.2, 0.2)
data = np.linspace(0, 1, height * width).reshape(height, width).astype('float32')

with rasterio.open(
    raster_path,
    'w',
    driver='GTiff',
    height=height,
    width=width,
    count=1,
    dtype='float32',
    crs='EPSG:4326',
    transform=transform,
) as dst:
    dst.write(data, 1)

raster_path


In [ ]:
# Pass raster to graph builder so node environmental covariates are sampled there
raster_paths = [raster_path]


In [ ]:
species = SpeciesData(
    name='demo',
    genotypes=genotypes,
    sample_coords=sample_coords,
)

graphs, stats = build_species_graphs(
    [species],
    mesh_spacing_km=80.0,
    project_to='EPSG:3857',
    coord_order='latlon',
    coords_crs='EPSG:4326',
    raster_paths=raster_paths,
    raster_fill_method='nearest',
    standardize=True,
)

model = train_model(graphs, hidden_dim=32, lr=1e-2, epochs=150, log_every=25)


In [ ]:
# Visualize learned graph edges (feature column index selectable)
ax, gdf = graphs[0].plot(
    edge_feature_idx=0,
    basemap=True,
    title=species.name,
)


In [ ]:
# Convert graph edges to a GeoDataFrame for custom plotting/export
ax, gdf = graphs[0].plot(
    edge_feature_idx=0,
    basemap=False,
    title=species.name,
)
gdf.head()
